# Fine-tune Gemma 4 on the Bangla homographs (free Google Colab)

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Run each cell from top to bottom (Shift+Enter). Read the note above each one.

The notebook starts in **QUICK TEST** mode (12 words, 2 epochs, a few minutes of training) so you can check that fine-tuning works. For the full run, change the lines marked `FULL` in steps 4 and 5.

Explanations: `HOW_TRAINING_WORKS.md` (English), `HOW_TRAINING_WORKS_bn.md` and `TAG_GUIDE_bn.md` (Bangla).

## 1. Check the GPU
You should see **Tesla T4** with about 15 GB. If it says *no GPU*, go back to the top and change the runtime.

In [ ]:
!nvidia-smi

## 2. Install the libraries (2-4 minutes)

In [ ]:
!pip install -q unsloth datasets

## 3. Upload `gemma_training.zip`
Click **Choose Files** and pick the zip from this project.

In [ ]:
from google.colab import files
import os, zipfile
os.chdir('/content')
up = files.upload()
name = list(up)[0]          # works even if the browser renamed it, e.g. 'gemma_training (1).zip'
zipfile.ZipFile(name).extractall('/content')
os.chdir('/content/gemma_training')
print('now in', os.getcwd())
print(sorted(os.listdir()))

## 4. Build the training files
**QUICK TEST** (default): 12 homograph words (about 240 sentences) + 200 plain sentences from Bangla Wikipedia.

For the **FULL** run, put `#` in front of the QUICK line and remove `#` from one FULL line. The tag options are explained in `TAG_GUIDE_bn.md`.

In [ ]:
!python prepare_data.py --words 12 --wiki 200                                   # QUICK TEST
# !python prepare_data.py --wiki 1500                                         # FULL, no tag
# !python prepare_data.py --wiki 1500 --tag "<hg>" --tag_mode trigger         # FULL, tag anywhere
# !python prepare_data.py --wiki 1500 --tag "<hg>" --tag_mode mark            # FULL, tag around the word

## 5. Fine-tune
QUICK TEST: 2 epochs, a few minutes. FULL: 3 epochs, roughly 20-40 minutes.
Watch `loss` go down.
If you see *CUDA out of memory*, open `train.py` (folder icon on the left), set `per_device_train_batch_size=4` and `gradient_accumulation_steps=4`, save, and run this cell again.

In [ ]:
!python train.py --epochs 2      # QUICK TEST
# !python train.py               # FULL

## 6. Try your own sentence

In [ ]:
!python test_examples.py --adapter gemma_homograph_lora --sentence "আসর নামাজের পর আমরা বাজারে যাব।"

## 7. Export one file to run on your own PC (10-20 minutes)
This merges Gemma and your training into a single **GGUF** file of about 3 GB.

In [ ]:
!python export_gguf.py
!ls -lh gemma_homograph_gguf

## 8. Download the results
Colab deletes everything when the session ends, so download now.
- `result.zip`: trained adapter and test data.
- the `.gguf` file: the model for your PC (about 3 GB; if the browser download fails, right-click it in the folder panel on the left and choose Download)

In [ ]:
!zip -q -r result.zip gemma_homograph_lora data
files.download('result.zip')
import glob
for g in glob.glob('gemma_homograph_gguf/*.gguf'):
    print('downloading', g)
    files.download(g)